# S04 — Solutions: NumPy

**Python for Applied Physics** | Master in Applied Physics  
⚠️ *Instructor reference — do not distribute before the exercise session.*

In [ ]:
import numpy as np
print(f"NumPy {np.__version__}")

---
## Exercise 1 — Gaussian beam intensity profile

In [ ]:
w0 = 300e-6
P  = 0.5

# 1. Radial grid
r = np.linspace(-3 * w0, 3 * w0, 1024)

# 2. Intensity
I0 = 2 * P / (np.pi * w0**2)
I  = I0 * np.exp(-2 * r**2 / w0**2)

# 3. Peak
I_peak        = I.max()
I_peak_formula = 2 * P / (np.pi * w0**2)

# 4. First index where I < I_peak/e² on the positive-r half
pos_half = r >= 0
r_pos    = r[pos_half]
I_pos    = I[pos_half]
i_hw     = np.argmax(I_pos < I_peak * np.exp(-2))   # first True → first crossing
# Map back to full array index
i_hw_full = np.where(pos_half)[0][i_hw]

# 5. Integrated power: P = 2π ∫ I(r) r dr
P_integrated = 2 * np.pi * np.trapz(I_pos * r_pos, r_pos)

assert I.shape == (1024,)
assert np.isclose(I_peak, I_peak_formula, rtol=1e-6)
assert np.isclose(np.abs(r[i_hw_full]), w0, rtol=0.02)
assert np.isclose(P_integrated, P, rtol=0.02)

print(f"Peak intensity    : {I_peak/1e4:.2f} W/cm²")
print(f"1/e² radius index : {i_hw_full}  → r = {np.abs(r[i_hw_full])*1e6:.1f} µm")
print(f"Integrated power  : {P_integrated*1e3:.2f} mW")
print("All assertions passed ✓")

---
## Exercise 2 — Boolean masking on a beam cross-section

In [ ]:
N_px   = 128
px     = 10e-6
w_beam = 200e-6
I0_2d  = 1.0

x_px = (np.arange(N_px) - N_px // 2) * px
y_px = (np.arange(N_px) - N_px // 2) * px

# 1. Broadcasting: X shape (1, N), Y shape (N, 1)
X    = x_px[np.newaxis, :]   # (1, 128)
Y    = y_px[:, np.newaxis]   # (128, 1)
I2d  = I0_2d * np.exp(-2 * (X**2 + Y**2) / w_beam**2)   # (128, 128)

# 2. Pixels above 1/e²
threshold = I0_2d * np.exp(-2)
n_above   = (I2d > threshold).sum()

# 3. Intensity-weighted centroid
# X and Y already broadcast to (128, 128) when multiplied with I2d
x_c = np.sum(I2d * X) / np.sum(I2d)
y_c = np.sum(I2d * Y) / np.sum(I2d)

# 4. Clip
I2d_clipped = np.where(I2d >= threshold, I2d, 0.0)

assert I2d.shape == (128, 128)
assert np.isclose(I2d[64, 64], I0_2d, rtol=1e-3)
assert n_above > 0
assert np.isclose(x_c, 0.0, atol=px)
assert np.isclose(y_c, 0.0, atol=px)
assert I2d_clipped.min() == 0.0

print(f"Pixels above 1/e² : {n_above}")
print(f"Centroid           : ({x_c*1e6:.2f}, {y_c*1e6:.2f}) µm")
print(f"Max after clipping : {I2d_clipped.max():.4f}")
print("All assertions passed ✓")

---
## Exercise 3 — 2D Gaussian field and radial profile extraction

In [ ]:
N   = 256
w   = 400e-6
E0  = 1.0

x = np.linspace(-3 * w, 3 * w, N)
y = np.linspace(-3 * w, 3 * w, N)

# 1. Complex 2D field via broadcasting
X    = x[np.newaxis, :]
Y    = y[:, np.newaxis]
E2d  = E0 * np.exp(-(X**2 + Y**2) / w**2)   # real-valued (phase = 0)

# 2. Intensity
I2d  = np.abs(E2d)**2

# 3. Horizontal cross-section
I_horiz = I2d[N//2, :]   # centre row

# 4. Diagonal via fancy indexing
diag_idx = np.arange(N)
I_diag   = I2d[diag_idx, diag_idx]

# 5. Verify
I_peak_2d = I2d[N//2, N//2]
idx_w     = np.argmin(np.abs(x - w))
I_at_w    = I_horiz[idx_w]

assert E2d.shape == (N, N)
assert np.isclose(I_peak_2d, E0**2, rtol=1e-3)
assert np.isclose(I_at_w / I_peak_2d, np.exp(-2), rtol=0.01)
assert I_diag.shape == (N,)

print(f"Peak intensity : {I_peak_2d:.4f}")
print(f"I(w) / I_peak  : {I_at_w/I_peak_2d:.4f}  (expected {np.exp(-2):.4f})")
print(f"Diagonal max   : {I_diag.max():.4f}")
print("All assertions passed ✓")

---
## Exercise 4 — Jones matrix chain

In [ ]:
def linear_polariser(angle_deg):
    θ = np.radians(angle_deg)
    c, s = np.cos(θ), np.sin(θ)
    return np.array([[c**2, c*s], [c*s, s**2]])

def half_wave_plate(fast_axis_deg):
    θ = np.radians(fast_axis_deg)
    c, s = np.cos(2*θ), np.sin(2*θ)
    return np.array([[c, s], [s, -c]])

def quarter_wave_plate(fast_axis_deg):
    """
    General retarder with δ = π/2.

    J = exp(-iπ/4) · [[cos²θ + i·sin²θ,  (1-i)·cosθ·sinθ],
                       [(1-i)·cosθ·sinθ,   sin²θ + i·cos²θ]]

    The global phase exp(-iπ/4) does not affect intensities.
    """
    θ  = np.radians(fast_axis_deg)
    c, s = np.cos(θ), np.sin(θ)
    δ  = np.pi / 2
    eiδ = np.exp(1j * δ)
    return np.exp(-1j * δ / 2) * np.array([
        [c**2 + eiδ * s**2, (1 - eiδ) * c * s],
        [(1 - eiδ) * c * s, s**2 + eiδ * c**2],
    ])

E_in = np.array([1.0 + 0j, 0.0 + 0j])

E1 = half_wave_plate(22.5)    @ E_in
E2 = quarter_wave_plate(0.0)  @ E1
E3 = linear_polariser(90)     @ E2

I_in = np.sum(np.abs(E_in)**2)
I1   = np.sum(np.abs(E1)**2)
I2   = np.sum(np.abs(E2)**2)
I3   = np.sum(np.abs(E3)**2)

assert np.isclose(I1, I_in, atol=1e-10)
assert np.isclose(I2, I_in, atol=1e-10)
assert np.isclose(np.abs(E1[0]), np.abs(E1[1]), atol=1e-6)
assert np.isclose(np.abs(E2[0]), np.abs(E2[1]), atol=1e-6)

print(f"I_in    : {I_in:.4f}")
print(f"After HWP: {I1:.4f}  E={E1}")
print(f"After QWP: {I2:.4f}  E={E2}")
print(f"Phase diff: {np.angle(E2[1]/E2[0])*180/np.pi:.1f}°  (expected ±90°)")
print(f"Transmission through vertical analyser: {I3:.4f}")
print("All assertions passed ✓")

---
## Exercise 5 — Pulse spectrum and time–bandwidth product

In [ ]:
def pulse_spectrum(t, E_t):
    N  = len(t)
    dt = t[1] - t[0]
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
    E_f  = np.fft.fftshift(np.fft.fft(E_t))
    return freq, np.abs(E_f)**2


def fwhm(x, y):
    """FWHM of a peaked 1D array."""
    y_norm = y / y.max()
    above  = y_norm >= 0.5
    if not above.any():
        return np.nan
    return x[above].max() - x[above].min()


N   = 4096
dt  = 5e-15
τ1  = 50e-15
τ2  = 200e-15

t = (np.arange(N) - N // 2) * dt

E_t1    = np.exp(-t**2 / (2 * τ1**2))
freq1, S1 = pulse_spectrum(t, E_t1)
Δt1 = fwhm(t, E_t1**2)
Δν1 = fwhm(freq1, S1)
TBP1 = Δt1 * Δν1

E_t2    = np.exp(-t**2 / (2 * τ2**2))
freq2, S2 = pulse_spectrum(t, E_t2)
Δt2 = fwhm(t, E_t2**2)
Δν2 = fwhm(freq2, S2)
TBP2 = Δt2 * Δν2

TBP_limit = 2 * np.log(2) / np.pi

assert np.isclose(TBP1, TBP_limit, rtol=0.01)
assert np.isclose(TBP2, TBP_limit, rtol=0.01)
assert Δt2 > Δt1
assert Δν2 < Δν1

print(f"τ = {τ1*1e15:.0f} fs:  Δt={Δt1*1e15:.1f} fs, Δν={Δν1/1e12:.2f} THz, TBP={TBP1:.4f}")
print(f"τ = {τ2*1e15:.0f} fs:  Δt={Δt2*1e15:.1f} fs, Δν={Δν2/1e12:.2f} THz, TBP={TBP2:.4f}")
print(f"Transform limit: {TBP_limit:.4f}")
print("All assertions passed ✓")

---
## Exercise 6 — Chirped pulse and group delay dispersion

In [ ]:
def pulse_spectrum(t, E_t):
    N  = len(t)
    dt = t[1] - t[0]
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
    E_f  = np.fft.fftshift(np.fft.fft(E_t))
    return freq, np.abs(E_f)**2

def fwhm(x, y):
    y_norm = y / y.max()
    above  = y_norm >= 0.5
    return x[above].max() - x[above].min() if above.any() else np.nan

N    = 4096
dt   = 5e-15
τ    = 50e-15
λ0   = 800e-9
c    = 3e8
ν0   = c / λ0
GDD  = 500e-30

t    = (np.arange(N) - N // 2) * dt
E_t  = np.exp(-t**2 / (2 * τ**2))

freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
ω    = 2 * np.pi * freq

# 1. FFT
E_f = np.fft.fftshift(np.fft.fft(E_t))

# 2. Apply GDD phase
phase_GDD   = 0.5 * GDD * ω**2
E_f_chirped = E_f * np.exp(1j * phase_GDD)

# 3. IFFT
E_t_chirped = np.fft.ifft(np.fft.ifftshift(E_f_chirped))
I_chirped   = np.abs(E_t_chirped)**2
Δt_chirped  = fwhm(t, I_chirped)

Δt_TL       = 2 * np.sqrt(2 * np.log(2)) * τ
Δt_expected = Δt_TL * np.sqrt(1 + (GDD / τ**2)**2)

# 4. Verify spectrum
_, S_original = pulse_spectrum(t, E_t)
_, S_chirped  = pulse_spectrum(t, E_t_chirped)

# 5. Instantaneous frequency
phase_t   = np.unwrap(np.angle(E_t_chirped))
dphase_dt = np.diff(phase_t) / dt
ν_inst    = -dphase_dt / (2 * np.pi)   # baseband: no ν0 term
t_mid     = 0.5 * (t[:-1] + t[1:])

# 6. Chirp rate via linear fit in central region
central          = np.abs(t_mid) < 2 * Δt_chirped
coeffs           = np.polyfit(t_mid[central], ν_inst[central], 1)
chirp_rate_measured = coeffs[0]
chirp_rate_expected = 1 / (2 * np.pi * GDD)

assert np.isclose(Δt_chirped, Δt_expected, rtol=0.05)
assert np.allclose(S_original / S_original.max(),
                   S_chirped  / S_chirped.max(),  atol=1e-6)

print(f"Transform-limited FWHM : {Δt_TL*1e15:.1f} fs")
print(f"Chirped FWHM (measured): {Δt_chirped*1e15:.1f} fs")
print(f"Chirped FWHM (expected): {Δt_expected*1e15:.1f} fs")
print(f"Spectrum preserved: {np.allclose(S_original/S_original.max(), S_chirped/S_chirped.max(), atol=1e-6)}")
print(f"Chirp rate (measured) : {chirp_rate_measured/1e24:.3f} ×10²⁴ Hz/s")
print(f"Chirp rate (expected) : {chirp_rate_expected/1e24:.3f} ×10²⁴ Hz/s")
print("All assertions passed ✓")

---
## Exercise 7 — Multi-element ABCD beam propagation

In [ ]:
λ  = 1030e-9
w0 = 1e-3

def free_space(d):
    return np.array([[1.0, d], [0.0, 1.0]])

def thin_lens(f):
    return np.array([[1.0, 0.0], [-1.0/f, 1.0]])

z_L1, f1 = 50e-3,  50e-3
z_L2, f2 = 150e-3, 200e-3
z_L3, f3 = 450e-3, 100e-3

z = np.linspace(0, 600e-3, 1000)
w = np.zeros_like(z)

ray_in = np.array([w0, 0.0])

def system_matrix(z_eval):
    """Cumulative ABCD matrix from z=0 to z_eval."""
    # Start at entrance
    M = np.eye(2)
    z_prev = 0.0

    # Build the list of (position, matrix) events
    events = [
        (z_L1, thin_lens(f1)),
        (z_L2, thin_lens(f2)),
        (z_L3, thin_lens(f3)),
    ]

    for z_lens, M_lens in events:
        if z_eval <= z_prev:
            break
        d = min(z_eval, z_lens) - z_prev
        M = free_space(d) @ M
        z_prev += d
        if z_eval >= z_lens:
            M = M_lens @ M
            z_prev = z_lens

    # Remaining propagation after last lens
    if z_eval > z_prev:
        M = free_space(z_eval - z_prev) @ M

    return M


for i, zi in enumerate(z):
    M_i    = system_matrix(zi)
    ray_i  = M_i @ ray_in
    w[i]   = np.abs(ray_i[0])

# Focus
mask_after_L3 = z > z_L3
z_focus = z[mask_after_L3][np.argmin(w[mask_after_L3])]
w_focus = w[mask_after_L3].min()

# Beam radius just before L3
idx_before_L3 = np.argmin(np.abs(z - (z_L3 - 1e-6)))
w_in_L3       = w[idx_before_L3]

w_f_expected = f3 * λ / (np.pi * w_in_L3)

assert free_space(0.1) is not None
assert thin_lens(0.1) is not None
assert w.shape == (1000,)
assert np.all(w >= 0)
assert w_focus < w0
assert np.abs(z_focus - (z_L3 + f3)) < 10e-3, \
    f"Focus at {z_focus*1e3:.1f} mm, expected near {(z_L3+f3)*1e3:.1f} mm"

print(f"Beam radius just before L3 : {w_in_L3*1e3:.2f} mm")
print(f"Focus position             : {z_focus*1e3:.1f} mm  (expected {(z_L3+f3)*1e3:.1f} mm)")
print(f"Focus radius (measured)    : {w_focus*1e6:.1f} µm")
print(f"Focus radius (expected)    : {w_f_expected*1e6:.1f} µm")
print("All assertions passed ✓")